# Week 17, Day 1: Counting steps

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w017d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
count = 0
for i in range(4):
    for j in range(4):
        count += 1
print(count)

count = 0
n = 64
while n > 1:
    n //= 2
    count += 1
print(count)

In [ ]:
import time

def total(n):
    s = 0
    for i in range(n):
        s += i
    return s

for n in (100_000, 200_000, 400_000):
    start = time.perf_counter()
    total(n)
    took = time.perf_counter() - start
    print(f"n = {n:>7}: {took * 1000:.2f} ms")

In [ ]:
import timeit

print(timeit.timeit("sorted(data)", setup="data = list(range(1000, 0, -1))", number=1000))

> **Key idea.** The **input size**, n, is the measure of how big the problem is: the length of a list or a string, the number of rows in a file, the number itself for something like `fib(n)`. Step counts are always "for an input of size n".

In [ ]:
def find(items, target):
    for i in range(len(items)):      # the loop runs n times
        if items[i] == target:       # 1 comparison each time
            return i
    return -1                        # 1 step at the end

print(find([4, 8, 15, 16, 23, 42], 99))

In [ ]:
def find_counting(items, target):
    comparisons = 0
    for i in range(len(items)):
        comparisons += 1
        if items[i] == target:
            return i, comparisons
    return -1, comparisons

data = list(range(0, 2000, 2))      # 1000 even numbers
print(find_counting(data, 0))       # found at once
print(find_counting(data, 998))     # found halfway
print(find_counting(data, 7))       # not there: checks them all

In [ ]:
def has_duplicate(items):
    comparisons = 0
    n = len(items)
    for i in range(n):
        for j in range(i + 1, n):
            comparisons += 1
            if items[i] == items[j]:
                return True, comparisons
    return False, comparisons

for n in (10, 20, 40, 80):
    print(n, has_duplicate(list(range(n))))

*Interactive figure on the website: Each kind of algorithm reacts to doubling n in its own way. Watch the ratios after pressing Double n, then slide n up to a million..*

In [ ]:
def pairs_count(n):
    count = 0
    for i in range(n):
        for j in range(n):
            count += 1
    return count

for n in (100, 200, 400):
    print(n, pairs_count(n), pairs_count(2 * n) / pairs_count(n))

In [ ]:
def has_dup_set(items):
    seen = set()
    steps = 0
    for x in items:
        steps += 1
        if x in seen:
            return True, steps
        seen.add(x)
    return False, steps

def has_dup_pairs(items):
    steps = 0
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            steps += 1
            if items[i] == items[j]:
                return True, steps
    return False, steps

for n in (100, 1000, 3000):
    data = list(range(n))
    print(n, has_dup_pairs(data)[1], has_dup_set(data)[1])

> **Common mistake.** Don't count steps for one small input and conclude one version "is faster". For tiny inputs, the n² version may even win because each of its steps is simpler. Algorithms are compared by how their counts **grow**.

### Exercise: Count the comparisons

Write `max_counting(items)` that finds the largest item of a non-empty list with a loop, counting every comparison `x > best` it makes, and returns `(largest, comparisons)`. Start `best` at the first item, and compare it with each of the others.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert max_counting([3, 9, 2]) == (9, 2)\n    assert max_counting([5]) == (5, 0)\n    assert max_counting(list(range(1000))) == (999, 999)")

### Exercise: Count by formula

Each function below gives the number of times the line `count += 1` runs in a loop, for a size `n`. Write them **without loops**, as formulas, so they're instant for any n.

- `a(n)`: `for i in range(n): for j in range(n): count += 1`
- `b(n)`: `for i in range(n): for j in range(i): count += 1`
- `c(n)`: `for i in range(n): count += 1` followed by `for j in range(n): count += 1`
- `d(n)`: `for i in range(n): for j in range(5): count += 1`

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    for n in (0, 1, 7, 100):\n        assert a(n) == n * n and b(n) == n * (n - 1) // 2 and c(n) == 2 * n and d(n) == 5 * n\n    assert \"for \" not in source and \"while \" not in source")

### Exercise: Halving

Write `halvings(n)` returning how many times you can do `n = n // 2` before `n` reaches 1 (for `n` of 1 or more), using a loop. Then answer: how many halvings for a million? Store your answer in `million`.

In [ ]:
million = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert halvings(1) == 0 and halvings(2) == 1 and halvings(64) == 6 and halvings(100) == 6\n    assert million == 19 and halvings(1_000_000) == 19")

### Exercise: The doubling test

Write `doubling_ratio(steps, n)` that takes a function `steps` (which returns a step count for a size) and returns `steps(2 * n) / steps(n)`. Then write `guess(steps)` that uses the ratio at n = 1000 to name the growth: `"constant"` if the ratio is below 1.05, `"log"` if below 1.5, `"linear"` if below 3, `"quadratic"` if below 6, and `"faster"` otherwise.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math\ndef test_values():\n    \"several\"\n    assert doubling_ratio(lambda n: n * n, 10) == 4\n    assert guess(lambda n: 7) == \"constant\"\n    assert guess(lambda n: math.log2(n)) == \"log\"\n    assert guess(lambda n: 3 * n + 20) == \"linear\"\n    assert guess(lambda n: n * (n - 1) // 2) == \"quadratic\"\n    assert guess(lambda n: n ** 3) == \"faster\"")

### Exercise: Items in common

Write `common_slow(a, b)` and `common_fast(a, b)`. Both return `(count, steps)`: how many items of list `a` are also in list `b` (count each item of `a` once, even if repeated in `a`; items in each list are all different in the checks), and how many steps were taken. `common_slow` compares each item of `a` with each item of `b`, counting one step per comparison and stopping the inner loop when it finds a match. `common_fast` first makes a set of `b` (count `len(b)` steps for that), then counts one step per item of `a` checked.

In [ ]:
# Write your code here


In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert common_slow([1, 2, 3], [3, 4]) == (1, 2 + 2 + 1)\n    assert common_fast([1, 2, 3], [3, 4]) == (1, 2 + 3)\n    a, b = list(range(0, 2000, 2)), list(range(0, 2000, 3))\n    assert common_slow(a, b)[0] == common_fast(a, b)[0] == 334\n    assert common_fast(a, b)[1] == 1667\n    assert common_slow(a, b)[1] > 400_000")

### Bug hunt: A counter that doesn't count

This should print `100` (the inner line runs 10 × 10 times), but prints `10`. Fix it.

In [ ]:
count = 0
for i in range(10):
    for j in range(10):
        pass
    count += 1
print(count)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"100\"\n    expect_output(\"100\")")

### Bug hunt: Timing the wrong thing

This tries to time `sorted(data)` but the printed time includes building the list of a million numbers. Move the lines so only the sort is timed. The program should still print `sorted` and then a time.

In [ ]:
import time

start = time.perf_counter()
data = list(range(1_000_000, 0, -1))
result = sorted(data)
took = time.perf_counter() - start
print("sorted")
print(f"{took:.4f} s")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order():\n    \"only the sort is timed\"\n    lines = [l.strip() for l in source.splitlines() if l.strip()]\n    i_data = next(i for i, l in enumerate(lines) if l.startswith(\"data =\"))\n    i_start = next(i for i, l in enumerate(lines) if l.startswith(\"start =\"))\n    i_sort = next(i for i, l in enumerate(lines) if l.startswith(\"result =\"))\n    i_took = next(i for i, l in enumerate(lines) if l.startswith(\"took =\"))\n    assert i_data < i_start < i_sort < i_took\n    assert output.startswith(\"sorted\")")

### Puzzle: Counting pairs

Arrange the lines so the function returns the number of pairs `(i, j)` with `i < j` it looks at, along with whether any pair adds up to `target`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
a = ___   # comparisons has_duplicate makes on a list of 10 different items
b = ___   # if an algorithm takes about n² steps and n goes from 1,000 to 3,000, by what factor do the steps grow?
c = ___   # how many halvings to get from 1024 down to 1?
d = ___   # steps in "for i in range(n): for j in range(3): ..." when n = 50 (count the inner body)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == 45 and b == 9 and c == 10 and d == 150")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: pair sums three ways

Given a list of different whole numbers and a target, is there a pair of different items that adds up to the target? Write three versions, each returning `(answer, steps)`:

- `pairs_all(items, target)`: compare every pair `i < j`, one step per pair looked at, stopping when one is found.
- `pairs_set(items, target)`: go through the items once with a set of the items seen so far; for each item `x`, one step, and check whether `target - x` is in the set before adding `x`.
- `pairs_sorted(items, target)`: sort a copy (count no steps for sorting), then use two positions, `lo` at the start and `hi` at the end: while `lo < hi`, take one step; if the two items add to the target, stop; if the sum is too small, move `lo` up; otherwise move `hi` down.

Then set `slowest` to the name of the function (a string) that takes by far the most steps on a list of 2,000 numbers where no pair works. (`pairs_sorted` looks cheapest here only because the sort isn't counted; tomorrow you'll see that sorting costs about n log n steps, more than `pairs_set` needs in total.)

In [ ]:
slowest = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_small():\n    \"small\"\n    assert pairs_all([1, 4, 6, 9], 10) == (True, 3)\n    assert pairs_set([1, 4, 6, 9], 10) == (True, 3)\n    assert pairs_sorted([9, 1, 6, 4], 10) == (True, 1)\n    assert pairs_all([1, 2], 10)[0] is False and pairs_set([], 3) == (False, 0)\ndef test_big():\n    \"big\"\n    data = list(range(0, 4000, 2))\n    assert pairs_all(data, 1)[1] == 2000 * 1999 // 2\n    assert pairs_set(data, 1) == (False, 2000)\n    r = pairs_sorted(data, 1)\n    assert r[0] is False and r[1] == 1999\n    assert slowest == \"pairs_all\"")

## Warm-up

The first loop runs its body 4 × 4 = 16 times. The second halves 64 until it reaches 1: 64, 32, 16, 8, 4, 2, 1 is 6 halvings. Make the 4 into 1,000 and the first becomes a million; make the 64 into a million and the second only becomes 20. That gap is what this week is about.

## Timing code

The simplest way to find out how fast code is: run it and time it. `time.perf_counter()` gives a time in seconds from a precise clock; the difference between two readings is how long the code between them took.

Run it a few times. The times change from run to run, but the pattern holds: double `n`, and the time roughly doubles.

For small, quick pieces of code a single timing is too noisy. The `timeit` module runs the code many times and gives the total:

## Why seconds mislead

Timings are real, and you'll use them all your life. But as a way to **compare algorithms** they have problems:

- **The computer matters.** A laptop from ten years ago, a phone, and a server give very different numbers for the same code.
- **Everything else matters.** Other programs running, the battery saver, whether the data is already in memory: run the same code twice and you get two times.
- **The language and details matter.** The same idea is quicker in one language than another, and quicker with one Python trick than another.
- **The size you tried matters most.** Code that wins on 100 items can lose badly on a million.

What we want is a measure that belongs to the **algorithm**, not to the computer. That's the number of **steps** it takes, as a formula in the size of its input.

## Counting steps

A **step** is any basic operation that takes a fixed, small amount of time whatever n is: an assignment, a comparison, an arithmetic operation, indexing a list, calling `len`. We don't fuss over exactly which count as one step and which as two; what matters is how the **total** grows with n.

Count the steps in this search, for a list of n items, when the target isn't there:

About n comparisons, plus a handful of extra steps. With 10 items, about 10; with a million, about a million.

To check a count, add a **counter** to the code: a variable that goes up by 1 each time the step you're counting happens. Return it alongside the answer, or print it.

The count depends on the input, not just its size: finding the first item takes 1 comparison, a missing item takes n. Day 5 is about these best and worst cases; for now, count the **worst** case, which is the guarantee.

Nested loops multiply. Here's a function that checks whether any two items in a list are the same, by comparing every pair:

For n items, item 0 is compared with n − 1 others, item 1 with n − 2, and so on down to 0, which adds up to **n(n − 1)/2**. For 80 items that's 3,160.

## What happens when n doubles

The single most useful question to ask of any step count: **what happens when n doubles?**

| Steps for size n | Steps for size 2n | Doubling n... |
|---|---|---|
| 5 (a fixed number) | 5 | ...changes nothing |
| about log₂ n (halving) | one more | ...adds one step |
| about n | about 2n | ...doubles the steps |
| about n² | about 4n² | ...multiplies the steps by 4 |
| about 2ⁿ | about 2ⁿ × 2ⁿ | ...squares the steps |

Step through the sizes and press **Double n** to see each pattern:

That's why, in the doubling table from the timing example, `total` took twice as long for twice the input: its steps are about n. If you double the input and the time goes up by four, suspect a nested loop.

## Comparing algorithms

Here are two ways to check a list for duplicates. The first compares every pair (about n²/2 comparisons). The second remembers what it has seen in a set (Week 10), so each item is looked at once (about n set operations):

At 3,000 items it's 4.5 million comparisons against 3,000 steps. Both are correct; one is about 1,500 times less work, and the gap keeps growing with n. Choosing the right algorithm matters far more than a faster computer.

## Practice

## Quiz

## Daily challenge

## Before you go

Take three functions you've written in earlier weeks and add a step counter to each. Run them on inputs of size 100, 200 and 400, and write down what happens to the counts each time the size doubles. Tomorrow: Big-O notation, the shorthand for exactly that pattern.